In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023508AFBFB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023508BCD2B0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage

message = HumanMessage(content="Hey, My name is jeef")
model.invoke([message])

AIMessage(content='Nice to meet you, Jeef. Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 42, 'total_tokens': 66, 'completion_time': 0.025062306, 'completion_tokens_details': None, 'prompt_time': 0.003231019, 'prompt_tokens_details': None, 'queue_time': 0.046475241, 'total_time': 0.028293325}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf9ba-9ede-7702-8917-85e21454589a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 24, 'total_tokens': 66})

In [5]:
from langchain_core.messages import AIMessage 

model.invoke([
    HumanMessage(content="Hey, My name is jeef, i am genai engineer"),
    AIMessage(content="Hi Jeef, it's nice to meet you. Is there something I can help you with or would you like to chat?"),
    HumanMessage(content="what is my name and what do i do?")
])

##LLM model able to remember message history

AIMessage(content="Your name is Jeef. \n\nYou mentioned that you are a Genai engineer. However, I think you might have meant to say 'Genai' as in 'Genie' or possibly 'Gene' (as in genetic) and 'engineer', or possibly 'genie engineer' as in a robot, but I couldn't find any information on this term 'Genai'.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 94, 'total_tokens': 173, 'completion_time': 0.180699119, 'completion_tokens_details': None, 'prompt_time': 0.005788981, 'prompt_tokens_details': None, 'queue_time': 0.045855248, 'total_time': 0.1864881}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf9ba-9ffc-7920-b00a-61a3eb359e23-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 79, 'total_tokens': 173})

##LLM model able to remember message history based on sessions /  MESSAGE HISTORY
messge hidtry class ,to keep in track of inputs and outputs, and store in some memry, 
Future interactions can use these messages and pass them to chain as part of  input

In [6]:
from langchain_community.chat_message_histories import  ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#session id to store specific chat history
store={}
def get_session_history(session_id:str):
    if session_id not in store: 
        store[session_id]  = ChatMessageHistory() 
    return store[session_id]

##to interact with llm model with chat history 

with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [7]:
config = {"configurable":{"session_id":"chat1"}}

In [8]:
response = with_message_history.invoke(
    [HumanMessage(content="Hey , my name is jeef")],
    config = config
)

response.content

"Nice to meet you, Jeef. How's your day going so far?"

In [9]:
response = with_message_history.invoke(
    [HumanMessage(content="Hey , what is my name")],
    config = config
)

response.content

'Your name is Jeef.'

In [10]:
config1 = {"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hey , what is my name")],
    config = config1
)

response.content

"I'm not aware of your name. I'm a text-based AI assistant, and our conversation just started. I don't have any prior information about you. If you'd like to share your name, I'd be happy to chat with you!"

In [11]:
##config-sessionId-chat1 - stores convo 
##config1-sessionId-chat2 - stores diff convo 
## different session-ids stores different context of convo

#---MESSAGE HISTORY AND REMEMBERING CONTEXT

In [12]:
##CHATPROMPT TEMPLATE

from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","you are helpful assistant, please provide all the answers for the user input questions"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model
#chain.invoke({"message":"Hey,my name is john"})

In [13]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [14]:
config = {"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hey,My name is Cena")],
    config = config
)
response.content

"Nice to meet you, Cena! How can I assist you today? Do you have any questions or topics you'd like to discuss?"

In [15]:
response = with_message_history.invoke(
    [HumanMessage(content="whats my name?")],
    config = config
)
response.content

'Your name is Cena.'

In [16]:
##ADDING MORE COMPLEXITY - GIVING MULTIPLE INPUTS 
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","you are helpful assistant, please provide all the answers for the user input questions in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model


In [17]:
chain.invoke({"messages":[HumanMessage(content="Hey,my name is John")],"language":"kannada"})

AIMessage(content='ನಿಮ್ಮ ಹೆಸರು ಜಾನ್ ಎಂದು ಕೇಳಿದೀರಿ. ಹಲೂ ನಿಮಗೆ ಸಹಾಯ ಮಾಡುತ್ತೇನೇ!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 59, 'total_tokens': 158, 'completion_time': 0.153938015, 'completion_tokens_details': None, 'prompt_time': 0.002969515, 'prompt_tokens_details': None, 'queue_time': 0.046685895, 'total_time': 0.15690753}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf9ba-a591-7e70-b523-f125a9523cde-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 99, 'total_tokens': 158})

In [18]:
##wrapping this complicated chain with message history class
with_message_history = RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [19]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="Hello, my name is john")],"language":"HIndi"},
    config = config
)
response.content

'नमस्ते जॉन, आपका स्वागत है! कैसे जा रहे हैं आप?'

In [20]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="Hello, whats my name?")],"language":"kannada"},
    config = config
)
response.content

'ನಿಮ್ಮ ಹೆಸರು ಜಾನ್ ಆಗಿದೆ.'

In [21]:
##MANAGING CHAT HISTORY
from langchain_core.messages import SystemMessage,trim_messages,AIMessage
trimmer = trim_messages(
    max_tokens = 40,
    strategy = "last",
    token_counter = model,
    include_system = True,
    allow_partial = False,
    start_on ="human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)


c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\langchain_core\language_models\base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
# ASSIGINING TRIMMER TO THE CHAIN => NEED runnables,itemgetter
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough 

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")| trimmer) | prompt | model
)
 
chain.invoke({
    "messages": messages + [HumanMessage(content="which is the icecream i like?")],
    "language":"english"
})


AIMessage(content="I don't know, you haven't told me. Would you like to share your favorite ice cream flavor with me?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 97, 'total_tokens': 122, 'completion_time': 0.055813586, 'completion_tokens_details': None, 'prompt_time': 0.005444657, 'prompt_tokens_details': None, 'queue_time': 0.045795573, 'total_time': 0.061258243}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf9bb-33ea-7543-adc8-02c5d2884f1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 25, 'total_tokens': 122})

In [25]:
#wrapping this in message history 


with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [31]:
config = {"configurable":{"session_id":"chat5"}}
response = with_message_history.invoke(
    {
     "messages": messages + [HumanMessage(content="Hello, what is my name")],
     "language":"english"
     },
    config = config
)
response.content

"Unfortunately, I don't know your name yet. You haven't told me. Would you like to share it with me?"